# Manual Logging

This notebook demonstrates manual experiment logging in machine learning workflows using scikit-learn.

It covers how to:
- Load and preprocess a dataset
- Perform hyperparameter tuning with grid search
- Save experiment results and models to disk
- Reload and evaluate the best model

It also discusses the strengths and weaknesses of manual logging, and provides examples of logging results to files for reproducibility and analysis.

In [1]:
!pip install ucimlrepo

In [2]:
from ucimlrepo import fetch_ucirepo

import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder

# Fetch Statlog (German Credit Data) dataset
credit_data = fetch_ucirepo(id=27)

# Features and targets
X = credit_data.data.features
y = credit_data.data.targets

# Categorical encoding for X and y
X = pd.get_dummies(X)
le = LabelEncoder()
y = le.fit_transform(y)

# Split dataset
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.4, random_state=42)

/anaconda/envs/azureml_py38/lib/python3.10/site-packages/sklearn/preprocessing/_label.py:114: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)


In [3]:
print(credit_data.metadata.additional_info.summary)

This file concerns credit card applications.  All attribute names and values have been changed to meaningless symbols to protect confidentiality of the data.
  
This dataset is interesting because there is a good mix of attributes -- continuous, nominal with small numbers of values, and nominal with larger numbers of values.  There are also a few missing values.


In [ ]:
print(credit_data.metadata.additional_info.variable_info)

In [4]:
credit_data.data.features

,A15,A14,A13,A12,A11,A10,A9,A8,A7,A6,A5,A4,A3,A2,A1
0,0,202.0,g,f,1,t,t,1.25,v,w,g,u,0.000,30.83,b
1,560,43.0,g,f,6,t,t,3.04,h,q,g,u,4.460,58.67,a
2,824,280.0,g,f,0,f,t,1.50,h,q,g,u,0.500,24.50,a
3,3,100.0,g,t,5,t,t,3.75,v,w,g,u,1.540,27.83,b
4,0,120.0,s,f,0,f,t,1.71,v,w,g,u,5.625,20.17,b
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
685,0,260.0,g,f,0,f,f,1.25,h,e,p,y,10.085,21.08,b
686,394,200.0,g,t,2,t,f,2.00,v,c,g,u,0.750,22.67,a
687,1,200.0,g,t,1,t,f,2.00,ff,ff,p,y,13.500,25.25,a
688,750,280.0,g,f,0,f,f,0.04,v,aa,g,u,0.205,17.92,b


In [ ]:
unique, counts = np.unique(y, return_counts=True)

dict(zip(unique, counts))

## 1. Simple Logging with Grid Search

Simple logging in machine learning involves manually recording key information about experiments, such as model parameters, metrics, and results. This can be done using basic file I/O operations to write logs to text files or CSVs.

Advantages of simple logging include:

- Easy to implement: Requires minimal setup and can be done with basic programming skills.
- Flexible: You can log any information you find relevant without being constrained by a specific framework.
- No external dependencies: Does not require additional libraries or tools beyond the standard library.

Disadvantages include:

- Prone to human error: Manual logging is susceptible to mistakes, such as forgetting to log certain metrics or parameters.
- Limited scalability: As experiments grow in complexity, managing logs manually can become cumbersome.
- Lack of structure: Without a standardized format, logs can be difficult to analyze and compare.

To implement simple logging in a scikit-learn workflow, you can follow these steps:

1. Set up a logging function that writes relevant information to a file.
2. Call this function at key points in your workflow, such as after training a model or evaluating its performance.
3. Include important details like hyperparameters, training metrics, and any other information that will help you understand the experiment later.



In [5]:
# Define the model
model = RandomForestClassifier(random_state=42)

# Define the hyperparameter grid
param_grid = {
    'n_estimators': [10, 50, 100],
    'max_depth': [5, 10, None],
    'min_samples_split': [2, 5, 10]
}

from sklearn.model_selection import GridSearchCV
# Perform Grid Search
grid_search = GridSearchCV(model,
                           param_grid,
                           cv=3, scoring='accuracy', return_train_score=True)
grid_search.fit(X_train, y_train)

# Log all run results
results_df = pd.DataFrame(grid_search.cv_results_)

# Print the top 5 results as a preview
print("Preview of the top 5 runs:")
results_df.head(5)

Preview of the top 5 runs:


,mean_fit_time,std_fit_time,mean_score_time,std_score_time,param_max_depth,param_min_samples_split,param_n_estimators,params,split0_test_score,split1_test_score,split2_test_score,mean_test_score,std_test_score,rank_test_score,split0_train_score,split1_train_score,split2_train_score,mean_train_score,std_train_score
0,0.026509,0.003846,0.005558,0.000944,5,2,10,"{'max_depth': 5, 'min_samples_split': 2, 'n_es...",0.869565,0.862319,0.869565,0.867150,0.003416,3,0.913043,0.913043,0.916667,0.914251,0.001708
1,0.099667,0.000558,0.010344,0.000648,5,2,50,"{'max_depth': 5, 'min_samples_split': 2, 'n_es...",0.840580,0.869565,0.826087,0.845411,0.018076,26,0.931159,0.916667,0.923913,0.923913,0.005917
2,0.206885,0.015395,0.017968,0.001173,5,2,100,"{'max_depth': 5, 'min_samples_split': 2, 'n_es...",0.840580,0.855072,0.833333,0.842995,0.009038,27,0.934783,0.934783,0.923913,0.931159,0.005124
3,0.024033,0.000709,0.004843,0.000301,5,5,10,"{'max_depth': 5, 'min_samples_split': 5, 'n_es...",0.876812,0.862319,0.876812,0.871981,0.006832,1,0.913043,0.909420,0.905797,0.909420,0.002958
4,0.095137,0.000622,0.009674,0.000193,5,5,50,"{'max_depth': 5, 'min_samples_split': 5, 'n_es...",0.855072,0.876812,0.833333,0.855072,0.017750,19,0.920290,0.920290,0.916667,0.919082,0.001708


In [6]:
import os

results_dir = "./results"

# Save the full results to a CSV file
try:
    results_df.to_csv(os.path.join(results_dir, "all_hyperparameter_runs_log.csv"), index=False)
    print("All run results have been logged and saved.")
except Exception as e:
    print("Error while saving results:")
    print(type(e).__name__, "-", e)

All run results have been logged and saved.


### Load back the best model

In [9]:
import pandas as pd
from sklearn.ensemble import RandomForestClassifier

# Load the results from the CSV file
results_df = pd.read_csv(os.path.join(results_dir, "all_hyperparameter_runs_log.csv"))


# Find the best model based on the 'rank_test_score' column (or the metric you prefer)
best_params = results_df.loc[results_df['rank_test_score'] == 1, 'params'].values[0]

# Print the best parameters
print("Best Parameters:")
print(best_params)

# Convert the string representation of the dictionary to an actual dictionary if needed
import ast
if isinstance(best_params, str):
    best_params = ast.literal_eval(best_params)

# Initialize the model with the best parameters
best_model = RandomForestClassifier(**best_params, random_state=42)

# Print the initialized model to verify
print("Initialized model with best parameters:")
print(best_model)

# Fit the model on the training data
best_model.fit(X_train, y_train)

# Evaluate the model on the test data
test_accuracy = best_model.score(X_test, y_test)
print(f"Test Accuracy: {test_accuracy:.4f}")

Best Parameters:
{'max_depth': 5, 'min_samples_split': 5, 'n_estimators': 10}
Initialized model with best parameters:
RandomForestClassifier(max_depth=5, min_samples_split=5, n_estimators=10,
                       random_state=42)
Test Accuracy: 0.8551


## Room for improvement
- The model itself is not always saved unless explicitly done
- Model name and metadata may be missing
- Only selected metrics are logged
- Each rerun can overwrite previous logs unless managed
- Manual effort required for consistency and completeness
- Not scalable for large or collaborative projects

## 1.2 Simple Logging with Grid Search into experiment folder

In this section, we will implement a simple logging mechanism that saves the best model and its parameters into a designated experiment folder. This approach allows us to keep track of our experiments without relying on external libraries or frameworks.

In [10]:
import datetime
timestamp = datetime.datetime.now().strftime("%Y-%m-%d_%H-%M-%S")

EXPERIMENT_NAME = f"credit_random_forest_{timestamp}"
print(EXPERIMENT_NAME)
EXPERIMENT_FOLDER = f"./results/{EXPERIMENT_NAME}"
print(EXPERIMENT_FOLDER)

import pathlib
pathlib.Path(EXPERIMENT_FOLDER).mkdir(parents=True, exist_ok=True)

credit_random_forest_2026-02-19_09-04-37
./results/credit_random_forest_2026-02-19_09-04-37


In [11]:
# Define the hyperparameter grid
param_grid = {
    'n_estimators': [10, 50, 100],
    'max_depth': [None, 10, 20],
    'min_samples_split': [2, 5, 10]
}

from sklearn.model_selection import GridSearchCV
# Perform Grid Search
grid_search = GridSearchCV(model,
                           param_grid,
                           cv=3, scoring='accuracy', return_train_score=True)
grid_search.fit(X_train, y_train)

# Log all run results
results_df = pd.DataFrame(grid_search.cv_results_)

# Save the full results to a CSV file
results_df.to_csv(f'{EXPERIMENT_FOLDER}/all_hyperparameter_runs_log.csv', index=False)

# Print a message to indicate successful logging
print("All run results have been logged and saved.")

# Print the top 5 results as a preview
print("Preview of the top 5 runs:")
results_df.head(5)

All run results have been logged and saved.
Preview of the top 5 runs:


,mean_fit_time,std_fit_time,mean_score_time,std_score_time,param_max_depth,param_min_samples_split,param_n_estimators,params,split0_test_score,split1_test_score,split2_test_score,mean_test_score,std_test_score,rank_test_score,split0_train_score,split1_train_score,split2_train_score,mean_train_score,std_train_score
0,0.025036,0.002479,0.004332,0.000123,None,2,10,"{'max_depth': None, 'min_samples_split': 2, 'n...",0.855072,0.891304,0.833333,0.859903,0.023912,13,0.992754,0.985507,0.992754,0.990338,0.003416
1,0.099544,0.002291,0.009406,0.000512,None,2,50,"{'max_depth': None, 'min_samples_split': 2, 'n...",0.847826,0.884058,0.855072,0.862319,0.015654,7,1.000000,0.996377,0.996377,0.997585,0.001708
2,0.204478,0.006250,0.015841,0.000619,None,2,100,"{'max_depth': None, 'min_samples_split': 2, 'n...",0.855072,0.884058,0.862319,0.867150,0.012316,1,1.000000,1.000000,1.000000,1.000000,0.000000
3,0.026593,0.004861,0.006376,0.002778,None,5,10,"{'max_depth': None, 'min_samples_split': 5, 'n...",0.847826,0.876812,0.855072,0.859903,0.012316,13,0.956522,0.956522,0.974638,0.962560,0.008540
4,0.228215,0.014055,0.021529,0.001667,None,5,50,"{'max_depth': None, 'min_samples_split': 5, 'n...",0.862319,0.869565,0.847826,0.859903,0.009038,13,0.974638,0.960145,0.971014,0.968599,0.006158


In [12]:
best_params = results_df.loc[results_df['rank_test_score'] == 1, 'params'].values[0]

# Print the best parameters
print("Best Parameters:")
print(best_params)

# Convert the string representation of the dictionary to an actual dictionary if needed
import ast
if isinstance(best_params, str):
    best_params = ast.literal_eval(best_params)

# Initialize the model with the best parameters
best_model = RandomForestClassifier(**best_params, random_state=42)

# Print the initialized model to verify
print("Initialized model with best parameters:")
print(best_model)

# Fit the model on the training data
best_model.fit(X_train, y_train)

# Evaluate the model on the test data
test_accuracy = best_model.score(X_test, y_test)
print(f"Test Accuracy: {test_accuracy:.4f}")

# Save the best model to a file
import joblib
joblib.dump(best_model, f'{EXPERIMENT_FOLDER}/best_model.pkl')

Best Parameters:
{'max_depth': None, 'min_samples_split': 2, 'n_estimators': 100}
Initialized model with best parameters:
RandomForestClassifier(random_state=42)
Test Accuracy: 0.8659


['./results/credit_random_forest_2026-02-19_09-04-37/best_model.pkl']

# 2. More complex logging with Grid Search

Here we will implement a more complex logging mechanism that includes:
- Saving the model itself
- Logging multiple metrics
- Managing experiment metadata

In [13]:
import datetime
import pathlib

def create_experiment_folder(prefix="credit_random_forest", base_dir="./results"):
    # Create timestamp
    timestamp = datetime.datetime.now().strftime("%Y-%m-%d_%H-%M-%S")
    
    # Build experiment name and folder path
    experiment_name = f"{prefix}_{timestamp}"
    experiment_folder = f"{base_dir}/{experiment_name}"
    
    # Create the folder
    pathlib.Path(experiment_folder).mkdir(parents=True, exist_ok=True)
    
    return experiment_name, experiment_folder


experiment_name, experiment_folder = create_experiment_folder()
print(experiment_name)
print(experiment_folder)

credit_random_forest_2026-02-19_09-05-47
./results/credit_random_forest_2026-02-19_09-05-47


In [14]:
import joblib
from sklearn.metrics import make_scorer, accuracy_score, f1_score, precision_score, recall_score
from sklearn.model_selection import GridSearchCV

# Define the hyperparameter grid
param_grid = {
    'n_estimators': [10, 50, 100],
    'max_depth': [None, 10, 20],
    'min_samples_split': [2, 5, 10]
}

# Define the scoring metrics
scoring = {
    'accuracy': 'accuracy',
    'f1_macro': make_scorer(f1_score, average='macro'),
    'precision_macro': make_scorer(precision_score, average='macro'),
    'recall_macro': make_scorer(recall_score, average='macro')
}

# Perform Grid Search
grid_search = GridSearchCV(
    model,
    param_grid,
    cv=3,
    scoring=scoring,
    refit='accuracy',  # Choose the metric to refit the model
    return_train_score=True
)
grid_search.fit(X_train, y_train)

# Log all run results
results_df = pd.DataFrame(grid_search.cv_results_)

# Add a column for the model name
results_df['model_name'] = model.__class__.__name__

# Save the full results to a CSV file
results_df.to_csv(f'{experiment_folder}/all_hyperparameter_runs_log.csv', index=False)

# Print the top 5 results as a preview
print("Preview of the top 5 runs:")
results_df.head(5)

Preview of the top 5 runs:


,mean_fit_time,std_fit_time,mean_score_time,std_score_time,param_max_depth,param_min_samples_split,param_n_estimators,params,split0_test_accuracy,split1_test_accuracy,...,split2_test_recall_macro,mean_test_recall_macro,std_test_recall_macro,rank_test_recall_macro,split0_train_recall_macro,split1_train_recall_macro,split2_train_recall_macro,mean_train_recall_macro,std_train_recall_macro,model_name
0,0.023627,0.000137,0.013170,0.002994,None,2,10,"{'max_depth': None, 'min_samples_split': 2, 'n...",0.855072,0.891304,...,0.833617,0.860582,0.027006,14,0.992655,0.986190,0.992642,0.990496,0.003044,RandomForestClassifier
1,0.106493,0.002063,0.016570,0.001469,None,2,50,"{'max_depth': None, 'min_samples_split': 2, 'n...",0.847826,0.884058,...,0.854801,0.862280,0.017060,7,1.000000,0.996774,0.996774,0.997849,0.001521,RandomForestClassifier
2,0.202085,0.004183,0.022895,0.000240,None,2,100,"{'max_depth': None, 'min_samples_split': 2, 'n...",0.855072,0.884058,...,0.862998,0.867149,0.013916,1,1.000000,1.000000,1.000000,1.000000,0.000000,RandomForestClassifier
3,0.024056,0.000532,0.011611,0.000453,None,5,10,"{'max_depth': None, 'min_samples_split': 5, 'n...",0.847826,0.876812,...,0.858207,0.859474,0.012994,21,0.955078,0.955852,0.972887,0.961272,0.008219,RandomForestClassifier
4,0.106408,0.002474,0.021334,0.003114,None,5,50,"{'max_depth': None, 'min_samples_split': 5, 'n...",0.862319,0.869565,...,0.850011,0.860701,0.008649,12,0.974718,0.959984,0.969661,0.968121,0.006113,RandomForestClassifier


In [16]:
# Save the best model to a file
best_model = grid_search.best_estimator_
#% TODO Save best model to experiment folder
joblib.dump(best_model, f'{experiment_folder}/best_model.pkl')
# Print details of the best model and its metrics
best_params = grid_search.best_params_
best_score = grid_search.best_score_
print(f"\nBest Model Name: {model.__class__.__name__}")
print(f"Best Parameters: {best_params}")
print(f"Best Cross-Validation Accuracy: {best_score}")


Best Model Name: RandomForestClassifier
Best Parameters: {'max_depth': None, 'min_samples_split': 2, 'n_estimators': 100}
Best Cross-Validation Accuracy: 0.8671497584541065


### Load the best model

In [17]:
import joblib
import os


# List all files in the models directory
model_files = [f for f in os.listdir(experiment_folder) if f.startswith('best_model') and f.endswith('.pkl')]

# Check if there are any saved model files
if model_files:
    # Sort the model files by timestamp (newest last)
    model_files.sort(reverse=True)

    # Get the most recent model file
    latest_model_file = model_files[0]
    model_path = os.path.join(experiment_folder, latest_model_file)

    # Load the most recent model
    best_model = joblib.load(model_path)
    print(f"Loaded the most recent model: '{latest_model_file}'")

    # Evaluate the model on the test data
    test_accuracy = best_model.score(X_test, y_test)
    print(f"Test Accuracy: {test_accuracy:.4f}")
else:
    print("No saved model files found.")

Loaded the most recent model: 'best_model.pkl'
Test Accuracy: 0.8659
